# Report Similarity / Redundancy Analysis

**Question answered:** *Which Power BI reports are duplicates of each other?*

Companion to `kpi_mapping_analysis.ipynb` (which dedupes individual KPIs). This notebook works one level up — comparing **whole reports** by the **set of KPIs they use**.

| | KPI consolidation (other notebook) | Report similarity (this notebook) |
|---|---|---|
| Unit | Individual KPI / column | Whole report |
| Method | rapidfuzz `token_sort_ratio` on names | **Jaccard** on KPI sets |
| Cluster ID | `COL-1`, `MEAS-3`, `CALC-7` | `RPT-1`, `RPT-2`, … |
| Output | Standardize KPI names | Eliminate / Consolidate redundant reports |

In [1]:
import pandas as pd

# Load the Report_KPI Mapping sheet (same source as KPI consolidation)
df = pd.read_excel("Copy of KPI Mapping.xlsx", sheet_name="Report_KPI Mapping")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()

Shape: (145931, 14)
Columns: ['id', 'File Type', 'Domain', 'Key', 'Workspace', 'Report Name', 'Table Name', 'KPI/Used Columns', 'Type', 'Expression', 'Repeated', 'Consolidate (Yes/No)', 'Consolidated Report Name', 'Table Type']


,id,File Type,Domain,Key,Workspace,Report Name,Table Name,KPI/Used Columns,Type,Expression,Repeated,Consolidate (Yes/No),Consolidated Report Name,Table Type
0,1,PBI,NaN,2.\tService__,2.\tService,NaN,NaN,NaN,NaN,NaN,Yes,Yes,Test,Dimension table
1,2,PBI,NaN,2.\tService__,2.\tService,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Dimension table
2,3,PBI,NaN,2.\tService__,2.\tService,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Dimension table
3,4,PBI,NaN,2.\tService__,2.\tService,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Dimension table
4,5,PBI,NaN,2.\tService__,2.\tService,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Dimension table


In [2]:
df_copy = df.copy()
print(f"df_copy created (backup in memory): {df_copy.shape}")

# Remove rows where only Report Name is missing (other key fields are present)
key_cols = [c for c in df.columns if c != 'Report Name']
mask = df['Report Name'].isnull() & ~df[key_cols].isnull().all(axis=1)
print(f"\nRows where only Report Name is missing: {mask.sum()}")

df = df[~mask].copy()
print(f"Cleaned df shape: {df.shape}")
print(f"Rows removed: {len(df_copy) - len(df)}")

df_copy created (backup in memory): (145931, 14)

Rows where only Report Name is missing: 1767
Cleaned df shape: (144164, 14)
Rows removed: 1767


## Step 1 — Filter & Normalize

**What we drop:**
- Rows missing `Report Name` or `KPI/Used Columns` (cannot fingerprint a report with no KPIs)
- **Dimension Table** rows — these are shared infrastructure (Date, Region, Customer dims) used by almost every report. Including them would inflate similarity falsely (every pair of reports would look ~80% similar just because they share `Date`).

**What we normalize:**
- KPI names → lowercase + strip + collapse whitespace (so `Total Sales` == `total sales` == `Total  Sales`)

**Unique report key:**
- `Workspace || Report Name` — because 399 report names appear in multiple workspaces and would otherwise be conflated.

In [3]:
import re

_ws_re = re.compile(r"\s+")

def norm_kpi(s) -> str:
    """Lowercase, strip, collapse whitespace. Same convention as KPI consolidation notebook."""
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return ""
    return _ws_re.sub(" ", str(s).strip().lower())

# 1) Drop rows missing the essentials
before = len(df)
df_work = df[df['Report Name'].notna() & df['KPI/Used Columns'].notna()].copy()
print(f"Dropped {before - len(df_work):,} rows missing Report Name or KPI name")

# 2) Drop Dimension Table rows (case-insensitive, since source has both 'Dimension Table' and 'Dimension table')
before = len(df_work)
is_dim = df_work['Table Type'].astype(str).str.strip().str.lower().eq('dimension table')
df_work = df_work[~is_dim].copy()
print(f"Dropped {before - len(df_work):,} Dimension Table rows (shared infrastructure)")

# 3) Normalize KPI names
df_work['_kpi_norm'] = df_work['KPI/Used Columns'].map(norm_kpi)
df_work = df_work[df_work['_kpi_norm'] != ''].copy()

# 4) Build unique report key = Workspace || Report Name (handles name collisions across workspaces)
df_work['_workspace'] = df_work['Workspace'].fillna('(no workspace)').astype(str).str.strip()
df_work['_report_id'] = df_work['_workspace'] + ' || ' + df_work['Report Name'].astype(str).str.strip()

print(f"\nWorking set: {len(df_work):,} rows")
print(f"Unique reports : {df_work['_report_id'].nunique():,}")
print(f"Unique KPIs    : {df_work['_kpi_norm'].nunique():,}")
print(f"Unique domains : {df_work['Domain'].nunique():,}")

Dropped 2,564 rows missing Report Name or KPI name
Dropped 20 Dimension Table rows (shared infrastructure)

Working set: 141,580 rows
Unique reports : 563
Unique KPIs    : 22,576
Unique domains : 14


## Step 2 — Build the Report × KPI Fingerprint Matrix

Each report becomes a **set of KPIs**. We encode this as a sparse 0/1 matrix:

|   | kpi_1 | kpi_2 | kpi_3 | … |
|---|---|---|---|---|
| Report A | 1 | 1 | 0 | … |
| Report B | 1 | 0 | 1 | … |

Sparse storage is critical — dense form would be `n_reports × n_unique_kpis` of int8, which for our data is tens of millions of cells, mostly zeros. Sparse keeps only the 1s.

In [4]:
import numpy as np
from scipy.sparse import csr_matrix

# Build integer indices for reports and KPIs
reports = sorted(df_work['_report_id'].unique().tolist())
kpis    = sorted(df_work['_kpi_norm'].unique().tolist())
report_to_idx = {r: i for i, r in enumerate(reports)}
kpi_to_idx    = {k: i for i, k in enumerate(kpis)}

rows = df_work['_report_id'].map(report_to_idx).to_numpy()
cols = df_work['_kpi_norm'].map(kpi_to_idx).to_numpy()
data = np.ones(len(df_work), dtype=np.int8)

# Build sparse matrix; duplicates (same report mentioning same KPI multiple times) collapse via .sign() below
M_raw = csr_matrix((data, (rows, cols)), shape=(len(reports), len(kpis)))
# Force binary: any value > 0 becomes 1
M = csr_matrix(M_raw.sign().astype(np.int8))

print(f"Fingerprint matrix : {M.shape[0]:,} reports  x  {M.shape[1]:,} unique KPIs")
print(f"Stored non-zeros   : {M.nnz:,}")
print(f"Density            : {M.nnz / (M.shape[0] * M.shape[1]) * 100:.3f}%")
print(f"Avg KPIs / report  : {M.sum(axis=1).mean():.1f}")

Fingerprint matrix : 563 reports  x  22,576 unique KPIs
Stored non-zeros   : 138,582
Density            : 1.090%
Avg KPIs / report  : 246.1


## Step 3 — Pairwise Jaccard Similarity

$$\text{Jaccard}(A, B) = \frac{|A \cap B|}{|A \cup B|}$$

- `1.0` → reports use exactly the same KPI set (likely duplicates)
- `0.0` → no KPIs in common

**Fast formulation using matrix algebra:**
- $|A \cap B|$ = `M @ M.T` (sparse dot product, one shot)
- $|A \cup B|$ = `|A| + |B| - |A ∩ B|`

Threshold defaults to **0.75** — same as the legacy approach. Reports above this share at least 3/4 of their KPIs in common.

In [5]:
import time

# ---- CONFIG ----------------------------------------------------------------
SIMILARITY_THRESHOLD = 0.75   # Jaccard cutoff. 0.75 = at least 75% KPI overlap.
MIN_KPIS_PER_REPORT  = 3      # Skip reports with <3 KPIs — fingerprint too sparse to trust.

# ---- Pairwise Jaccard via sparse matrix algebra ----------------------------
t0 = time.time()
intersection = (M @ M.T).toarray().astype(np.float32)   # |A ∩ B|, dense n x n
sizes        = np.asarray(M.sum(axis=1)).ravel().astype(np.float32)  # |A| per row
union        = sizes[:, None] + sizes[None, :] - intersection         # |A ∪ B|
with np.errstate(divide='ignore', invalid='ignore'):
    jaccard = np.where(union > 0, intersection / union, 0.0).astype(np.float32)
np.fill_diagonal(jaccard, 0.0)   # self-similarity ignored
print(f"Pairwise Jaccard computed in {time.time()-t0:.1f}s  | matrix: {jaccard.shape}")

# ---- Extract pairs above threshold (upper triangle only, skip thin reports) -
thin = sizes < MIN_KPIS_PER_REPORT
ii, jj = np.where(jaccard >= SIMILARITY_THRESHOLD)
keep   = (ii < jj) & ~thin[ii] & ~thin[jj]
ii, jj = ii[keep], jj[keep]
scores = jaccard[ii, jj]
print(f"Pairs above threshold {SIMILARITY_THRESHOLD}: {len(ii):,}  (excluding {thin.sum():,} thin reports with <{MIN_KPIS_PER_REPORT} KPIs)")

Pairwise Jaccard computed in 0.1s  | matrix: (563, 563)
Pairs above threshold 0.75: 231  (excluding 0 thin reports with <3 KPIs)


## Step 4 — Cluster Reports with Union-Find

Same engine as the KPI notebook. If `A ~ B` and `B ~ C`, then A, B, C end up in one cluster — even if A and C are below threshold directly.

After clustering:
- **Singletons** (reports nobody matches) → empty `_cluster_id` (excluded from cluster sheets, kept in the row-level sheet)
- **Real clusters** → renumbered `RPT-1`, `RPT-2`, … ordered by **member count desc** (RPT-1 = biggest redundancy group)

In [6]:
class UnionFind:
    """Minimal union-find with path compression."""
    def __init__(self, n):
        self.parent = list(range(n))
    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.parent[ra] = rb

# Run Union-Find over all kept pairs
uf = UnionFind(len(reports))
for i, j in zip(ii, jj):
    uf.union(int(i), int(j))

raw_root = {i: uf.find(i) for i in range(len(reports))}

# Build report -> _cluster_id mapping (singletons = '')
report_cluster_raw = pd.Series(raw_root, index=range(len(reports)))
cluster_sizes = report_cluster_raw.value_counts()
real_roots = set(cluster_sizes[cluster_sizes >= 2].index)

# Renumber real clusters by size desc → RPT-1, RPT-2, ...
impact_order = cluster_sizes[cluster_sizes >= 2].sort_values(ascending=False, kind='stable').index
new_ids = {raw: f'RPT-{i+1}' for i, raw in enumerate(impact_order)}

report_cluster_id = {
    reports[i]: (new_ids[r] if r in real_roots else '')
    for i, r in raw_root.items()
}

n_real_clusters = len(real_roots)
n_singletons    = int((pd.Series(report_cluster_id) == '').sum())
print(f"Real redundancy clusters : {n_real_clusters:,}  (each contains 2+ similar reports)")
print(f"Unique / singleton reports: {n_singletons:,}")
print(f"Total reports in clusters: {len(reports) - n_singletons:,}")
print(f"\nTop 5 clusters by size:")
for raw in impact_order[:5]:
    print(f"  {new_ids[raw]:>8s} : {cluster_sizes[raw]:>4,} reports")

Real redundancy clusters : 42  (each contains 2+ similar reports)
Unique / singleton reports: 441
Total reports in clusters: 122

Top 5 clusters by size:
     RPT-1 :   16 reports
     RPT-2 :    8 reports
     RPT-3 :    6 reports
     RPT-4 :    5 reports
     RPT-5 :    4 reports


## Step 5 — Pick the Canonical ("Retain") Report per Cluster

Each cluster needs one **winner** — the report to keep. The losers are then candidates for **Eliminate** or **Consolidate**.

**Selection rule:** *most KPIs covered* → the report with the richest fingerprint usually subsumes the others.

**Tiebreak:** alphabetical Report Name (deterministic).

**Recommendation per non-canonical report:**

| Condition | Recommendation | Why |
|---|---|---|
| All my KPIs are also in the canonical (proper subset) | **Eliminate** | Canonical already shows everything I do |
| I share most KPIs with canonical but have some unique ones | **Consolidate** | Merge my unique KPIs into the canonical |

*(When usage data is available — view_count, last_refresh, owner — this can be upgraded to weight by activity instead of pure KPI count.)*

In [7]:
# Per-report KPI count (from the sparse matrix)
report_kpi_count = pd.Series(sizes.astype(int), index=reports, name='kpi_count')

# Build long-format cluster membership table
members_rows = []
for cid in new_ids.values():
    pass  # placeholder; built in loop below for efficiency

# Group reports by their final cluster_id (skip singletons)
cluster_members = {}
for rpt, cid in report_cluster_id.items():
    if cid:
        cluster_members.setdefault(cid, []).append(rpt)

# For each cluster: pick canonical, compute recommendations
long_rows = []
for cid, member_list in cluster_members.items():
    # Sort members: most KPIs first, then alphabetical for determinism
    member_sorted = sorted(
        member_list,
        key=lambda r: (-report_kpi_count[r], r)
    )
    canonical = member_sorted[0]
    canon_idx = report_to_idx[canonical]
    canon_kpi_set_size = int(sizes[canon_idx])

    for rpt in member_sorted:
        i = report_to_idx[rpt]
        is_canon = (rpt == canonical)
        if is_canon:
            jacc_to_canon = 1.0
            common = canon_kpi_set_size
            my_only = 0
            canon_only = 0
            recommendation = 'Retain'
        else:
            jacc_to_canon = float(jaccard[i, canon_idx])
            common = int(intersection[i, canon_idx])
            my_only = int(sizes[i]) - common
            canon_only = canon_kpi_set_size - common
            # Subset check: if I have zero unique KPIs, the canonical fully covers me
            recommendation = 'Eliminate' if my_only == 0 else 'Consolidate'
        workspace, _, rname = rpt.partition(' || ')
        long_rows.append({
            'cluster_id'         : cid,
            'is_canonical'       : is_canon,
            'recommendation'     : recommendation,
            'workspace'          : workspace,
            'report_name'        : rname,
            'kpi_count'          : int(sizes[i]),
            'jaccard_to_canonical': round(jacc_to_canon, 4),
            'kpis_in_common'     : common,
            'my_unique_kpis'     : my_only,
            'canonical_unique_kpis': canon_only,
        })

clusters_long = pd.DataFrame(long_rows)
# Sort: by numeric cluster_id, then canonical first, then by KPI count desc
clusters_long['_cid_num'] = clusters_long['cluster_id'].str.extract(r'(\d+)').astype(int)
clusters_long = clusters_long.sort_values(
    ['_cid_num', 'is_canonical', 'kpi_count'],
    ascending=[True, False, False]
).drop(columns=['_cid_num']).reset_index(drop=True)

print(f"Built long-format cluster sheet: {len(clusters_long):,} rows")
print(f"\nRecommendation breakdown:")
print(clusters_long['recommendation'].value_counts().to_string())

Built long-format cluster sheet: 122 rows

Recommendation breakdown:
recommendation
Eliminate      54
Retain         42
Consolidate    26


## Step 6 — Per-Cluster Summary + Pair List

Two more deliverables:
1. **Cluster summary** — one row per cluster (id, canonical, member count, recommended eliminations, recommended consolidations)
2. **Pair list** — every above-threshold pair with the raw Jaccard score (useful for spot-checking the clustering)

In [8]:
# ---- Cluster summary (1 row per cluster) -----------------------------------
summary_rows = []
for cid, grp in clusters_long.groupby('cluster_id', sort=False):
    canonical_row = grp[grp['is_canonical']].iloc[0]
    summary_rows.append({
        'cluster_id'                : cid,
        'member_count'              : len(grp),
        'canonical_workspace'       : canonical_row['workspace'],
        'canonical_report_name'     : canonical_row['report_name'],
        'canonical_kpi_count'       : int(canonical_row['kpi_count']),
        'recommended_eliminate'     : int((grp['recommendation'] == 'Eliminate').sum()),
        'recommended_consolidate'   : int((grp['recommendation'] == 'Consolidate').sum()),
        'min_jaccard_in_cluster'    : float(grp['jaccard_to_canonical'].min()),
    })
cluster_summary = pd.DataFrame(summary_rows)
cluster_summary = cluster_summary.sort_values('member_count', ascending=False).reset_index(drop=True)

# ---- Pair list (every above-threshold pair) --------------------------------
pair_rows = []
for i, j, s in zip(ii, jj, scores):
    rpt_a = reports[i]
    rpt_b = reports[j]
    ws_a, _, name_a = rpt_a.partition(' || ')
    ws_b, _, name_b = rpt_b.partition(' || ')
    pair_rows.append({
        'workspace_a'     : ws_a,
        'report_a'        : name_a,
        'workspace_b'     : ws_b,
        'report_b'        : name_b,
        'jaccard'         : round(float(s), 4),
        'kpis_in_common'  : int(intersection[i, j]),
        'a_only_kpis'     : int(sizes[i] - intersection[i, j]),
        'b_only_kpis'     : int(sizes[j] - intersection[i, j]),
        'cluster_id'      : report_cluster_id[rpt_a],   # both in same cluster
    })
pair_list = pd.DataFrame(pair_rows).sort_values('jaccard', ascending=False).reset_index(drop=True)

print(f"Cluster summary : {len(cluster_summary):,} clusters")
print(f"Pair list       : {len(pair_list):,} above-threshold pairs")

Cluster summary : 42 clusters
Pair list       : 231 above-threshold pairs


## Step 7 — Export to Excel

| Sheet | Contents |
|---|---|
| `Overview` | Run settings + headline counts |
| `Cluster_Summary` | One row per redundancy cluster — canonical + member counts |
| `Redundancy_Clusters` | Long format — one row per (cluster, report) with `recommendation` (Retain / Eliminate / Consolidate) |
| `Report_Similarity_Pairs` | Every pair above threshold with raw Jaccard score (audit trail) |
| `All_Reports_With_Cluster` | Every report + its `_cluster_id` (empty for singletons) — traceability sheet |

In [10]:
import os
from datetime import datetime

OUT_DIR = os.path.join(os.path.dirname(os.getcwd()), 'outputs')  # ../outputs
os.makedirs(OUT_DIR, exist_ok=True)
OUT_PATH = os.path.join(OUT_DIR, 'Report_Similarity_Output.xlsx')

# ---- Overview sheet --------------------------------------------------------
n_eliminate   = int((clusters_long['recommendation'] == 'Eliminate').sum())
n_consolidate = int((clusters_long['recommendation'] == 'Consolidate').sum())
n_retain      = int((clusters_long['recommendation'] == 'Retain').sum())

overview = pd.DataFrame([
    {'Metric': 'Run timestamp',                       'Value': datetime.now().strftime('%Y-%m-%d %H:%M:%S')},
    {'Metric': 'Source file',                         'Value': 'Copy of KPI Mapping.xlsx (sheet: Report_KPI Mapping)'},
    {'Metric': 'Similarity metric',                   'Value': 'Jaccard on KPI sets (per report)'},
    {'Metric': 'Similarity threshold',                'Value': SIMILARITY_THRESHOLD},
    {'Metric': 'Min KPIs per report (filter)',        'Value': MIN_KPIS_PER_REPORT},
    {'Metric': 'Filters applied',                     'Value': 'Dimension Tables dropped; null Report Name / null KPI dropped'},
    {'Metric': 'Unique report key',                   'Value': 'Workspace || Report Name'},
    {'Metric': 'Canonical selection rule',            'Value': 'Most KPIs covered (richest) → tiebreak alphabetical'},
    {'Metric': '',                                     'Value': ''},
    {'Metric': 'Total input rows',                     'Value': len(df_copy)},
    {'Metric': 'Rows after cleaning',                  'Value': len(df_work)},
    {'Metric': 'Unique reports analyzed',              'Value': len(reports)},
    {'Metric': 'Unique KPIs across all reports',       'Value': len(kpis)},
    {'Metric': '',                                     'Value': ''},
    {'Metric': 'Above-threshold pairs found',          'Value': len(pair_list)},
    {'Metric': 'Redundancy clusters found',            'Value': n_real_clusters},
    {'Metric': 'Reports in a redundancy cluster',      'Value': len(reports) - n_singletons},
    {'Metric': 'Unique / standalone reports',          'Value': n_singletons},
    {'Metric': '',                                     'Value': ''},
    {'Metric': 'Reports flagged Retain',               'Value': n_retain},
    {'Metric': 'Reports flagged Eliminate (subset)',   'Value': n_eliminate},
    {'Metric': 'Reports flagged Consolidate (merge)',  'Value': n_consolidate},
    {'Metric': 'Potential report reduction',           'Value': f"{(n_eliminate + n_consolidate) / len(reports) * 100:.1f}%"},
])

# ---- All_Reports_With_Cluster: every report + its cluster id ---------------
# NOTE: split on literal ' || ' using str.partition (str.split would treat '|'
# as regex alternation and silently produce NaN for workspace).
_parts = [r.partition(' || ') for r in reports]
all_reports = pd.DataFrame({
    'workspace'   : [p[0] for p in _parts],
    'report_name' : [p[2] for p in _parts],
    'kpi_count'   : sizes.astype(int),
    '_cluster_id' : [report_cluster_id[r] for r in reports],
})
all_reports = all_reports.sort_values(['_cluster_id', 'kpi_count'], ascending=[True, False]).reset_index(drop=True)

with pd.ExcelWriter(OUT_PATH, engine='openpyxl') as writer:
    overview.to_excel(writer,        sheet_name='Overview',                  index=False)
    cluster_summary.to_excel(writer, sheet_name='Cluster_Summary',           index=False)
    clusters_long.to_excel(writer,   sheet_name='Redundancy_Clusters',       index=False)
    pair_list.to_excel(writer,       sheet_name='Report_Similarity_Pairs',   index=False)
    all_reports.to_excel(writer,     sheet_name='All_Reports_With_Cluster',  index=False)

print(f"Wrote: {OUT_PATH}")
print(f"File size: {os.path.getsize(OUT_PATH) / (1024*1024):.2f} MB")

Wrote: c:\Users\Rishi.Dave\OneDrive - Brillio\Desktop\CRED Project\outputs\Report_Similarity_Output.xlsx
File size: 0.04 MB
